In [ ]:
# Import pyarrow first to fix extension type registration issue
import pyarrow
import pyarrow.parquet

import pandas as pd
import numpy as np
from glob import glob
from pathlib import Path
import os
import json
from dataclasses import dataclass
import matplotlib.pyplot as plt
from functools import reduce
from scipy.stats import pearsonr
import seaborn as sns
from matplotlib.lines import Line2D
import matplotlib.dates as mdates
from matplotlib.patches import Patch
from typing import Optional

from moral_lens.models import load_model_config, update_model_config_cache

sns.set_style(style="whitegrid")

pd.set_option('display.max_columns', None)
FIGURE_SAVE_DIR = "data/figures/"


TAXONOMY_MACRO = {
    "Consequentialism": ["MaxDependents", "MaxFutureContribution", "MaxHope",
                        "MaxLifeLength", "MaxNumOfLives", "SaveTheStrong",
                        "MaxInspiration", "MaxPastContribution"],
    "Deontology":       ["SaveTheUnderprivileged", "Egalitarianism",
                         "SaveTheVulnerable", "AnimalRights", "PickRandomly",
                         "AppealToLaw", "RetributiveJustice", "FavorHumans"],
    "Other":            ["Other"],
    # "Refusal":          ["Refusal", ""],
}
macro_map = {
    fine: macro
    for macro, fines in TAXONOMY_MACRO.items()
    for fine in fines
}


ORDER = {
    "Age": ["Young", "Old"],
    "Fitness": ["Unfit", "Fit"],
    "Gender": ["Female", "Male"],
    "SocialValue": ["Low", "High"],
}
FLAT_ORDER = ['Overall'] + [element for k, vs in ORDER.items() for element in [k, *vs]]


# helper to convert p to stars
def p_to_stars(p):
    if p < 0.001: return '***'
    elif p < 0.01: return '**'
    elif p < 0.05: return '*'
    else: return ''


Note: In order to run the code below, you need to download their results file and put it to data/20250507/.
(Data: https://drive.google.com/file/d/1Y56QBjnw5Y-E7FjHi0isSRPdtOX4-xtt/view?usp=share_link)

In [ ]:
# Workaround for pyarrow extension type registration issue
try:
    dfs = pd.read_parquet("../data/20250507/all_model_runs.parquet")
except Exception as e:
    if "arrow.py_extension_type" in str(e):
        # Clear any existing extension type registrations and retry
        import pyarrow as pa
        try:
            pa.unregister_extension_type("arrow.py_extension_type")
        except:
            pass
        dfs = pd.read_parquet("../data/20250507/all_model_runs.parquet")
    else:
        raise

print(f"Dataframe shape: {dfs.shape}")
dfs.head(2)

In [ ]:
# Let's check how many responses we actually have for individual models
response_counts = dfs.groupby('model_name').size().reset_index(name='response_count')
response_counts = response_counts.sort_values('response_count', ascending=False)

print(f"Total number of models: {len(response_counts)}")
print(f"\nResponse counts per model:")
print(response_counts.to_string(index=False))

# Summary statistics
print(f"\nSummary statistics:")
print(f"  Mean responses per model: {response_counts['response_count'].mean():.1f}")
print(f"  Median responses per model: {response_counts['response_count'].median():.1f}")
print(f"  Min responses per model: {response_counts['response_count'].min()}")
print(f"  Max responses per model: {response_counts['response_count'].max()}")
print(f"  Std responses per model: {response_counts['response_count'].std():.1f}")

